# Phase 2.5: Geometry Preprocessing & UV Displacement Dataset Engine

**Objective:** Transform raw high-resolution 3D scans into paired 1024x1024 UV displacement maps, position maps, normal maps, and facial validity masks.

**Hardware Constraint:** This notebook runs entirely on a **CPU-only Kaggle Session (0 GPU quota consumed)**.

**Pipeline Steps:**
1. Ray-casting from neutral FLAME base mesh along outward vertex normals to intersect scan surfaces.
2. Barycentric UV triangle rasterization over FLAME UV layout at 1024x1024 resolution.
3. Empirical p99 measurement across scan corpus.
4. Lossless 16-bit uint PNG storage (d_norm in [-1, 1] -> [0, 65535]).
5. Dataset packaging for Stage 3 Detail GAN training.


In [ ]:
# ── CELL 1: Environment & Dependency Setup (CPU Session) ─────────────────────────
import os
import sys
from pathlib import Path

# Clone repository if running standalone on Kaggle
if not Path("scripts/build_uv_displacement_dataset.py").exists():
    !git clone https://github.com/NetPranav/Humanoid-Face-3D.git /kaggle/working/Humanoid-Face-3D
    %cd /kaggle/working/Humanoid-Face-3D

!pip install trimesh opencv-python Pillow pyyaml tqdm scipy --quiet

import trimesh
import cv2
import numpy as np
import json

print(f"Trimesh version: {trimesh.__version__}")
print(f"OpenCV version:  {cv2.__version__}")


In [ ]:
# ── CELL 2: Discover Attached Raw Scans & FLAME Model Assets ────────────────────
import shutil
from pathlib import Path

SCAN_DIR = None
input_dir = Path("/kaggle/input")
if input_dir.exists():
    for p in input_dir.glob("*"):
        if "flame-model" not in p.name.lower() and p.is_dir():
            if any(p.glob("**/*.obj")):
                SCAN_DIR = p
                print(f"Found external scan dataset at: {SCAN_DIR}")
                break

FLAME_PATH = Path("/kaggle/input/flame-model/generic_model.pkl")
if not FLAME_PATH.exists():
    FLAME_PATH = Path("./data/flame_model/generic_model.pkl")

# If no external scan dataset attached, use head_template.obj as baseline scan
if SCAN_DIR is None or not any(SCAN_DIR.glob("**/*.obj")):
    sample_scan_dir = Path("/tmp/sample_scans/subject_001")
    sample_scan_dir.mkdir(parents=True, exist_ok=True)
    head_template = Path("/kaggle/input/flame-model/head_template.obj")
    if head_template.exists():
        shutil.copy(head_template, sample_scan_dir / "scan.obj")
        SCAN_DIR = Path("/tmp/sample_scans")
        print(f"[Notice] Using template scan {head_template} as baseline scan.")
    elif Path("./data/flame_model/head_template.obj").exists():
        shutil.copy("./data/flame_model/head_template.obj", sample_scan_dir / "scan.obj")
        SCAN_DIR = Path("/tmp/sample_scans")
    else:
        SCAN_DIR = Path("./data/raw_scans")

OUTPUT_DIR = Path("/kaggle/working/uv_displacement_dataset_1024")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw Scans Directory: {SCAN_DIR.resolve()} (Exists: {SCAN_DIR.exists()})")
print(f"FLAME Model Path:    {FLAME_PATH.resolve()} (Exists: {FLAME_PATH.exists()})")
print(f"Output Dataset Path: {OUTPUT_DIR.resolve()}")


In [ ]:
# ── CELL 3: Run Geometry Preprocessing Engine (1024x1024) ───────────────────────
!python scripts/build_uv_displacement_dataset.py \
    --scan_dir {SCAN_DIR} \
    --flame_model {FLAME_PATH} \
    --output_dir {OUTPUT_DIR} \
    --resolution 1024


In [ ]:
# ── CELL 4: Verify Normalization Statistics & Lossless Round-Trip ───────────────
stats_file = OUTPUT_DIR / "normalization_stats.json"
if stats_file.exists():
    with open(stats_file) as f:
        stats = json.load(f)
    print("\n--- Measured Normalization Statistics ---")
    p99 = stats.get("p99_mm", "N/A")
    res = stats.get("resolution", 1024)
    num_s = stats.get("num_samples", 0)
    fmt = stats.get("storage_format")
    print(f"Displacement p99: {p99} mm")
    print(f"Resolution:       {res}x{res}")
    print(f"Sample Count:     {num_s}")
    print(f"Storage Format:   {fmt}")
else:
    print(f"Statistics file not found at: {stats_file}")


In [ ]:
# ── CELL 5: Visualize Preprocessed UV Maps (Displacement, Normal, Mask) ────────
import matplotlib.pyplot as plt
from scripts.build_uv_displacement_dataset import decode_displacement_16bit

disp_files = sorted(list(OUTPUT_DIR.glob("*_disp.png")))
if disp_files:
    sample_file = disp_files[0]
    print(f"Visualizing sample: {sample_file.name}")
    u16_disp = cv2.imread(str(sample_file), cv2.IMREAD_UNCHANGED)
    norm_disp = decode_displacement_16bit(u16_disp)
    plt.figure(figsize=(6, 6))
    plt.imshow(norm_disp, cmap="inferno", vmin=-1.0, vmax=1.0)
    plt.title(f"{sample_file.stem} (Normalized Displacement [-1, 1])")
    plt.colorbar(label="Normalized Magnitude")
    plt.axis("off")
    plt.savefig(str(OUTPUT_DIR / "sample_preview.png"))
    plt.show()
else:
    print("No preprocessed displacement maps found to display.")
